# Clause Astra GPU Training
Первый облачный GPU smoke-run для Clause Astra. Ноутбук не создаёт runtime автоматически: открой его в Colab, выбери **Runtime → Change runtime type → GPU**, затем запусти ячейки по порядку.

В первом запуске используем 100 шагов и Qwen3-0.6B-Base. Это проверка пайплайна LoRA, а не финальное обучение модели с нуля.

In [ ]:
import os, sys, json, pathlib, subprocess, urllib.request
import torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU не подключён. В Colab выбери Runtime → Change runtime type → GPU.')
p = torch.cuda.get_device_properties(0)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(p.total_memory / 1024**3, 2))

## 1. Подключи архив датасета
Сначала помести `ClauseAstra1_Dataset_v1.zip` в Google Drive. Архив по отчёту проекта весит около 164 MB. Если в архиве уже есть отдельные train/validation JSONL, ноутбук найдёт их автоматически.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ARCHIVE = '/content/drive/MyDrive/ClauseAstra1_Dataset_v1.zip'
EXTRACT_DIR = '/content/clause-astra-dataset'
if pathlib.Path(ARCHIVE).exists():
    import zipfile
    pathlib.Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as z:
        bad = z.testzip()
        if bad: raise RuntimeError(f'Повреждён файл архива: {bad}')
        z.extractall(EXTRACT_DIR)
    print('Архив распакован:', EXTRACT_DIR)
else:
    print('Архив не найден. Положи его в MyDrive под именем ClauseAstra1_Dataset_v1.zip и повтори ячейку.')

In [ ]:
root = pathlib.Path(EXTRACT_DIR)
jsonl_files = sorted(root.rglob('*.jsonl')) if root.exists() else []
for f in jsonl_files:
    print(f'{f}  |  {f.stat().st_size:,} bytes')
train_candidates = [f for f in jsonl_files if 'train' in f.name.lower()]
valid_candidates = [f for f in jsonl_files if any(x in f.name.lower() for x in ('valid','val','dev'))]
TRAIN_JSONL = str(train_candidates[0]) if train_candidates else ''
VALIDATION_JSONL = str(valid_candidates[0]) if valid_candidates else ''
print('\nTRAIN_JSONL =', TRAIN_JSONL or 'НУЖНО ВЫБРАТЬ ВРУЧНУЮ')
print('VALIDATION_JSONL =', VALIDATION_JSONL or 'НУЖНО ВЫБРАТЬ ВРУЧНУЮ')
# Если автоматический выбор неверен, укажи пути вручную в двух строках ниже:
# TRAIN_JSONL = '/content/clause-astra-dataset/path/to/train.jsonl'
# VALIDATION_JSONL = '/content/clause-astra-dataset/path/to/validation.jsonl'

In [ ]:
assert TRAIN_JSONL and pathlib.Path(TRAIN_JSONL).is_file(), 'Не найден train JSONL — укажи TRAIN_JSONL вручную.'
assert VALIDATION_JSONL and pathlib.Path(VALIDATION_JSONL).is_file(), 'Не найден validation JSONL — укажи VALIDATION_JSONL вручную.'
print('Train bytes:', pathlib.Path(TRAIN_JSONL).stat().st_size)
print('Validation bytes:', pathlib.Path(VALIDATION_JSONL).stat().st_size)
for path in (TRAIN_JSONL, VALIDATION_JSONL):
    with open(path, encoding='utf-8') as f:
        first = next((line for line in f if line.strip()), '')
    sample = json.loads(first)
    print(path, 'sample keys:', list(sample.keys()))

## 2. Установи библиотеки и запусти короткий LoRA run
PyTorch из Colab не переустанавливается — так мы не ломаем готовую CUDA-сборку. Результат и логи сохраняются в Google Drive.

In [ ]:
!pip -q install 'transformers>=4.51.0' 'datasets>=3.0.0' 'peft>=0.15.0' 'accelerate>=1.0.0' safetensors
SCRIPT_URL = 'https://raw.githubusercontent.com/arshinovseva13-ux/CloseApps/main/clause-gpu-cloud/training/train_lora.py'
SCRIPT = '/content/train_clause_astra_lora.py'
urllib.request.urlretrieve(SCRIPT_URL, SCRIPT)
OUTPUT_DIR = '/content/drive/MyDrive/ClauseAstra/checkpoints/run-001'
os.makedirs(OUTPUT_DIR, exist_ok=True)
cmd = [sys.executable, SCRIPT, '--train', TRAIN_JSONL, '--validation', VALIDATION_JSONL,
       '--base-model', 'Qwen/Qwen3-0.6B-Base', '--max-steps', '100',
       '--max-length', '512', '--output-dir', OUTPUT_DIR, '--save-steps', '50']
print('Запуск:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    raise RuntimeError(f'Training завершился с кодом {result.returncode}')
print('Готово. Checkpoint:', OUTPUT_DIR)

## 3. Что проверить после первого запуска
- `final-adapter/` содержит LoRA-веса и tokenizer.
- `final_metrics.json` содержит train/eval метрики.
- `run_metadata.json` фиксирует базовую модель, GPU и параметры.
- Сгенерируй несколько тестовых ответов и проверь качество отдельно от loss.

Не запускай длинное обучение, пока smoke-run не прошёл без ошибок и адаптер не сохранён.